In [6]:
import ROOT
import math
import os
import pandas as pd

%jsroot on

# ============================================================
# USER SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Lung_ICRP/lung.root"
tree_name = "t"

selected_volume = 2
selected_process = 2013       # Compton process
selected_pdg = 22             # gamma

E0 = 662.0                    # incident gamma energy, keV

# Incident beam direction:
# /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Optional step selection
use_step_selection = False
selected_step = 0

# Histogram settings
n_theta_bins = 90             # 2-degree bins
n_energy_bins = 140           # 5-keV bins

theta_min = 0.0
theta_max = 180.0

energy_min = 0.0
energy_max = 700.0

# Output files
csv_file = "tissue_theta_et_Eprime_events.csv"
summary_csv_file = "tissue_theta_et_Eprime_binned_summary.csv"



# ============================================================
# OPEN ROOT FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"ROOT file was not found:\n{file_path}"
    )

root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise RuntimeError(
        f"Could not open ROOT file:\n{file_path}"
    )

tree = root_file.Get(tree_name)

if not tree:
    root_file.ls()
    raise RuntimeError(
        f"Tree '{tree_name}' was not found."
    )

print("ROOT file opened successfully")
print("Tree entries:", tree.GetEntries())


# ============================================================
# CHECK REQUIRED BRANCHES
# ============================================================

required_branches = [
    "pdg", "pro", "vlm",
    "et", "k",
    "px", "py", "pz"
]

if use_step_selection:
    required_branches.append("stp")

available_branches = {
    branch.GetName()
    for branch in tree.GetListOfBranches()
}

missing_branches = [
    branch for branch in required_branches
    if branch not in available_branches
]

if missing_branches:
    raise RuntimeError(
        "Missing branches: " + ", ".join(missing_branches)
    )

print("All required branches are available.")


# ============================================================
# NORMALIZE INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(
    ix**2 + iy**2 + iz**2
)

if incident_norm <= 0:
    raise ValueError(
        "Incident direction cannot be a zero vector."
    )

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm


# ============================================================
# REMOVE OLD ROOT OBJECTS
# ============================================================

for object_name in [
    "h_theta_et",
    "h_theta_Eprime",
    "h_theta_Eprime_from_et",
    "c_theta_energy"
]:
    old_object = ROOT.gROOT.FindObject(object_name)

    if old_object:
        if old_object.InheritsFrom("TCanvas"):
            old_object.Close()
        else:
            old_object.Delete()


# ============================================================
# CREATE HISTOGRAMS
# ============================================================

# theta versus et
h_theta_et = ROOT.TH2F(
    "h_theta_et",
    "Scattering angle versus et;"
    "Scattering angle #theta (degrees);"
    "et (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

# theta versus E' taken directly from k
h_theta_Eprime = ROOT.TH2F(
    "h_theta_Eprime",
    "Scattering angle versus scattered photon energy E';"
    "Scattering angle #theta (degrees);"
    "E' = k (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

# theta versus E0-et, useful as a consistency check
h_theta_Eprime_from_et = ROOT.TH2F(
    "h_theta_Eprime_from_et",
    "Scattering angle versus E_{0}-et;"
    "Scattering angle #theta (degrees);"
    "E_{0}-et (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

for histogram in [
    h_theta_et,
    h_theta_Eprime,
    h_theta_Eprime_from_et
]:
    histogram.SetStats(0)


# ============================================================
# EVENT LOOP
# ============================================================

rows = []

matching_records = 0
valid_records = 0
zero_momentum_records = 0
invalid_records = 0

for event_number, event in enumerate(tree):

    branch_lengths = [
        len(event.pdg),
        len(event.pro),
        len(event.vlm),
        len(event.et),
        len(event.k),
        len(event.px),
        len(event.py),
        len(event.pz)
    ]

    if use_step_selection:
        branch_lengths.append(len(event.stp))

    n = min(branch_lengths)

    for i in range(n):

        # ----------------------------------------------------
        # EVENT SELECTION
        # ----------------------------------------------------

        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pro[i]) != selected_process:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        if use_step_selection:
            if int(event.stp[i]) != selected_step:
                continue

        matching_records += 1

        # ----------------------------------------------------
        # OUTGOING PHOTON DIRECTION
        # ----------------------------------------------------

        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        momentum = math.sqrt(
            px**2 + py**2 + pz**2
        )

        if momentum <= 0:
            zero_momentum_records += 1
            continue

        ux = px / momentum
        uy = py / momentum
        uz = pz / momentum

        # Incident direction dot scattered direction
        cos_theta = (
            ix * ux +
            iy * uy +
            iz * uz
        )

        # Protect acos against numerical roundoff
        cos_theta = max(
            -1.0,
            min(1.0, cos_theta)
        )

        theta_deg = math.degrees(
            math.acos(cos_theta)
        )

        # ----------------------------------------------------
        # ENERGY VALUES
        # ----------------------------------------------------

        et_value = float(event.et[i])

        # For a photon, kinetic energy k equals photon energy
        Eprime_k = float(event.k[i])

        # Use this only when et represents transferred energy
        Eprime_from_et = E0 - et_value

        values_to_check = [
            theta_deg,
            et_value,
            Eprime_k,
            Eprime_from_et
        ]

        if not all(math.isfinite(v) for v in values_to_check):
            invalid_records += 1
            continue

        # ----------------------------------------------------
        # FILL HISTOGRAMS
        # ----------------------------------------------------

        h_theta_et.Fill(
            theta_deg,
            et_value
        )

        h_theta_Eprime.Fill(
            theta_deg,
            Eprime_k
        )

        h_theta_Eprime_from_et.Fill(
            theta_deg,
            Eprime_from_et
        )

        # ----------------------------------------------------
        # SAVE ONE ROW PER SELECTED RECORD
        # ----------------------------------------------------

        rows.append({
            "event_number": event_number,
            "record_index": i,
            "vlm": int(event.vlm[i]),
            "pro": int(event.pro[i]),
            "pdg": int(event.pdg[i]),
            "theta_deg": theta_deg,
            "et_keV": et_value,
            "Eprime_k_keV": Eprime_k,
            "Eprime_from_E0_minus_et_keV": Eprime_from_et,
            "difference_k_minus_E0_minus_et_keV":
                Eprime_k - Eprime_from_et,
            "px": px,
            "py": py,
            "pz": pz
        })

        valid_records += 1


# ============================================================
# CONVERT TO DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

print("\nSelection:")
print("vlm =", selected_volume)
print("pro =", selected_process)
print("pdg =", selected_pdg)

if use_step_selection:
    print("stp =", selected_step)

print("\nMatching records:", matching_records)
print("Valid exported records:", valid_records)
print("Zero-momentum records:", zero_momentum_records)
print("Invalid records:", invalid_records)

if df.empty:
    raise RuntimeError(
        "No valid records were found for the selected conditions."
    )


# ============================================================
# CREATE ANGLE-BINNED SUMMARY
# ============================================================

theta_bin_width = (
    theta_max - theta_min
) / n_theta_bins

df["theta_bin_left_deg"] = (
    theta_min +
    ((df["theta_deg"] - theta_min) // theta_bin_width)
    * theta_bin_width
)

df["theta_bin_right_deg"] = (
    df["theta_bin_left_deg"] +
    theta_bin_width
)

df["theta_bin_center_deg"] = (
    df["theta_bin_left_deg"] +
    theta_bin_width / 2.0
)

summary_df = (
    df.groupby(
        [
            "theta_bin_left_deg",
            "theta_bin_right_deg",
            "theta_bin_center_deg"
        ],
        as_index=False
    )
    .agg(
        counts=("theta_deg", "size"),
        mean_et_keV=("et_keV", "mean"),
        std_et_keV=("et_keV", "std"),
        mean_Eprime_k_keV=("Eprime_k_keV", "mean"),
        std_Eprime_k_keV=("Eprime_k_keV", "std"),
        mean_Eprime_from_et_keV=(
            "Eprime_from_E0_minus_et_keV",
            "mean"
        )
    )
)

summary_df = summary_df.sort_values(
    "theta_bin_center_deg"
)


# ============================================================
# SAVE CSV FILES
# ============================================================

df.to_csv(
    csv_file,
    index=False
)

summary_df.to_csv(
    summary_csv_file,
    index=False
)


print("\nFiles saved:")
print(csv_file)
print(summary_csv_file)


# ============================================================
# DRAW THREE PLOTS
# ============================================================

c_theta_energy = ROOT.TCanvas(
    "c_theta_energy",
    "Theta, et and scattered photon energy",
    1600,
    1200
)

c_theta_energy.Divide(2, 2)


# ------------------------------------------------------------
# Plot 1: theta versus et
# ------------------------------------------------------------

pad1 = c_theta_energy.cd(1)

pad1.SetLeftMargin(0.11)
pad1.SetRightMargin(0.15)
pad1.SetBottomMargin(0.12)

h_theta_et.Draw("COLZ")


# ------------------------------------------------------------
# Plot 2: theta versus E' directly from k
# ------------------------------------------------------------

pad2 = c_theta_energy.cd(2)

pad2.SetLeftMargin(0.11)
pad2.SetRightMargin(0.15)
pad2.SetBottomMargin(0.12)

h_theta_Eprime.Draw("COLZ")


# ------------------------------------------------------------
# Plot 3: theta versus E0-et
# ------------------------------------------------------------

pad3 = c_theta_energy.cd(3)

pad3.SetLeftMargin(0.11)
pad3.SetRightMargin(0.15)
pad3.SetBottomMargin(0.12)

h_theta_Eprime_from_et.Draw("COLZ")


# ------------------------------------------------------------
# Plot 4: theoretical scattered-energy curve
# ------------------------------------------------------------

pad4 = c_theta_energy.cd(4)

pad4.SetLeftMargin(0.12)
pad4.SetRightMargin(0.05)
pad4.SetBottomMargin(0.12)

me = 511.0

graph_theory = ROOT.TGraph()
graph_theory.SetName("graph_theory")

for j in range(181):

    theta_theory = float(j)

    theta_rad = math.radians(
        theta_theory
    )

    Eprime_theory = (
        E0 /
        (
            1.0 +
            (E0 / me) *
            (1.0 - math.cos(theta_rad))
        )
    )

    graph_theory.SetPoint(
        j,
        theta_theory,
        Eprime_theory
    )

graph_theory.SetTitle(
    "Theoretical Compton scattered energy;"
    "Scattering angle #theta (degrees);"
    "E' (keV)"
)

graph_theory.SetLineWidth(3)
graph_theory.Draw("AL")


# ============================================================
# DISPLAY
# ============================================================

c_theta_energy.Modified()
c_theta_energy.Update()
c_theta_energy.Draw()

ROOT file opened successfully
Tree entries: 1000000
All required branches are available.

Selection:
vlm = 2
pro = 2013
pdg = 22

Matching records: 14226
Valid exported records: 14226
Zero-momentum records: 0
Invalid records: 0

Files saved:
tissue_theta_et_Eprime_events.csv
tissue_theta_et_Eprime_binned_summary.csv
